In [0]:
# ── Cell 1: Load raw CSV ──────────────────────────────────────────────────
from pyspark.sql import functions as F

storage_account = "onlinelearningd1"
key = dbutils.secrets.get(scope="adls-scope", key="storage-key")
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", key)

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"abfss://raw@{storage_account}.dfs.core.windows.net/online_learning_dirty.csv")
)

print(f"Raw row count: {df_raw.count()}")
print(f"Columns: {df_raw.columns}")
df_raw.printSchema()

Raw row count: 1225
Columns: ['user_id', 'age', 'gender', 'country', 'education_level', 'hours_spent_weekly', 'preferred_device', 'course_completed', 'satisfaction_rating', 'login_frequency', 'quiz_score_avg']
root
 |-- user_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- hours_spent_weekly: string (nullable = true)
 |-- preferred_device: string (nullable = true)
 |-- course_completed: string (nullable = true)
 |-- satisfaction_rating: string (nullable = true)
 |-- login_frequency: string (nullable = true)
 |-- quiz_score_avg: string (nullable = true)



In [0]:
# ── Cell 2: Data quality report ───────────────────────────────────────────
print("=== NULL COUNTS ===")
null_counts = df_raw.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c)).alias(c)
    for c in df_raw.columns
])
display(null_counts)

print("\n=== DUPLICATE ROWS ===")
total     = df_raw.count()
distinct  = df_raw.dropDuplicates().count()
print(f"Total: {total}, Distinct: {distinct}, Duplicates: {total - distinct}")

print("\n=== SAMPLE DIRTY VALUES ===")
print("Gender unique:", [r[0] for r in df_raw.select("gender").distinct().collect()])
print("Education unique:", [r[0] for r in df_raw.select("education_level").distinct().collect()])
print("Course completed unique:", [r[0] for r in df_raw.select("course_completed").distinct().collect()])
print("Device unique:", [r[0] for r in df_raw.select("preferred_device").distinct().collect()])


=== NULL COUNTS ===


user_id,age,gender,country,education_level,hours_spent_weekly,preferred_device,course_completed,satisfaction_rating,login_frequency,quiz_score_avg
0,0,0,36,0,96,0,0,59,24,74



=== DUPLICATE ROWS ===
Total: 1225, Distinct: 1204, Duplicates: 21

=== SAMPLE DIRTY VALUES ===
Gender unique: ['  Other  ', 'F', '  F  ', 'Man', 'm', 'f', 'Female', '  Female  ', 'female', 'M', 'other', 'Other', 'others', 'MALE', 'Woman', 'Others', 'male', 'OTHER', 'Non-binary', 'Male', 'FEMALE', '  Male  ']
Education unique: ['Masters', '  Other  ', 'B.Sc', 'High School', 'Highschool', '  Bachelor  ', 'doctorate', 'ph.d', 'master', '  High School  ', 'PhD', '  PhD  ', 'Other', 'HIGH SCHOOL', 'Master', 'B.Tech', 'BACHELOR', 'Bachelors', 'P.H.D', 'PHD', 'M.Tech', 'high school', 'High school', 'Phd', 'bachelor', 'Bachelor', 'MSc', 'M.S', '  Master  ', '  M.S  ']
Course completed unique: ['0', 'False', 'YES', 'No', 'Yes', 'false', '1', 'no', 'True', 'yes', 'NO', 'true']
Device unique: ['PC', 'iPad', 'desktop', 'MOBILE', 'phone', 'mobile', 'DESKTOP', 'tablet', 'Phone', '  Tablet  ', 'laptop', '  Mobile  ', '  Desktop  ', '  Phone  ', 'Laptop', 'Tab', 'tab', 'TABLET', 'Mobile', 'Tablet', 

In [0]:
# ── Cell 3: Save to bronze (raw Parquet — no changes yet) ─────────────────
(df_raw.write
    .mode("overwrite")
    .parquet(f"abfss://silver@{storage_account}.dfs.core.windows.net/bronze_snapshot/")
)
